# 12 - Stored Procedures & Procedural SQL for Data Engineers

**The complete guide to Stored Procedures in modern Data Engineering:**
- **What they are & why they exist:** encapsulating multi-step DDL/DML, business logic, and control flow inside the warehouse.
- **The Modern Lakehouse Reality:** Databricks SQL (SQL Scripting & Unity Catalog procedures), Snowflake (SQL/Python procs), BigQuery, and legacy RDBMS migrations (Oracle / T-SQL).
- **Core Patterns Drilled:**
  1. **Anatomy & Parameters:** `IN`, `OUT`, variables (`DECLARE`), and branching (`IF / ELSE`).
  2. **Idempotent Batch Reloads:** Atomic partition overwrites and MERGE upserts.
  3. **Transactions & Error Handling:** `BEGIN TRANSACTION`, `COMMIT`, `ROLLBACK`, and `TRY...CATCH`.
  4. **Dynamic SQL (`EXECUTE IMMEDIATE`):** Dynamic table/partition maintenance & SQL injection risks.
  5. **Modernization & Refactoring:** Eliminating legacy RDBMS cursors and loops into vectorized PySpark / modern SQL.


In [1]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "src" / "de_helpers.py").exists())
sys.path.insert(0, str(ROOT / "src"))

from de_helpers import (
    q, q1, spark_q, assert_grain, tables, columns, explain, display, fs_ls,
    get_spark, get_dbutils, spark_read, connect
)
from pyspark.sql import functions as F, Window

spark = get_spark()
dbutils = get_dbutils()
print("Setup complete. Local DuckDB warehouse and Spark session ready.")


Setup complete. Local DuckDB warehouse and Spark session ready.


### 1. The Big Picture: Stored Procedure vs UDF vs View vs dbt / Python

In Data Engineering interviews, the most common question is:
> *"When would you write a Stored Procedure instead of a View, a UDF, or an external script in Python/dbt?"*

| Construct | Purpose | Modifies Data (DML)? | Control Flow (IF/WHILE)? | Returns | Execution Engine |
| :--- | :--- | :--- | :--- | :--- | :--- |
| **View** | Query shortcut / encapsulation | ❌ No (read-only) | ❌ No | Virtual table | SQL engine |
| **User-Defined Function (UDF)** | Computes a reusable transformation | ❌ No (pure function, no side-effects) | ⚠️ Limited | Single scalar value or table | SQL / Spark |
| **Stored Procedure** | Automates multi-step workflows & side-effects | ✅ **Yes** (`INSERT`, `UPDATE`, `DELETE`, `MERGE`) | ✅ **Yes** (`IF`, `LOOP`, `TRY...CATCH`) | Status code, OUT params, or Result Sets | Database engine |
| **dbt / PySpark** | Declarative transformations & distributed pipelines | ✅ Yes | ✅ Yes (Jinja / Python) | Tables / Views / Delta tables | Spark / Warehouse |

#### Where do Stored Procedures fit today?
1. **Legacy Enterprises (Oracle, SQL Server, Teradata):** Massive suites of business logic live in stored procedures. Data Engineers frequently maintain or **migrate** them to Databricks/Snowflake.
2. **Atomic Ingestion & Guardrails:** Executing multi-table operations with ACID transaction guarantees (`COMMIT` / `ROLLBACK`) directly within the database.
3. **Database-level Access Control:** Giving an application user `EXECUTE` permission on a procedure without giving them raw `INSERT`/`UPDATE` access to underlying tables.
4. **Modern Cloud Data Warehouses:**
   - **Snowflake:** Supports both SQL scripting and Python/Java stored procedures (Snowpark).
   - **Databricks:** Supports SQL User-Defined Routines, `EXECUTE IMMEDIATE`, and Python Notebooks orchestrated via Databricks Workflows.


---
### Module 1: Anatomy & Control Flow (Parameters, Variables, IF/ELSE)

A stored procedure typically consists of:
1. **Parameters:** `IN` (inputs), `OUT` (outputs returned to caller), `INOUT` (both).
2. **Variable Declaration:** `DECLARE v_variable_name TYPE;`
3. **Control Flow:** `IF condition THEN ... ELSE ... END IF;`
4. **DML / Query Operations:** Selecting, inserting, or returning result sets.

#### Standard ANSI / Databricks SQL Syntax:
```sql
CREATE OR REPLACE PROCEDURE get_provider_summary(
    IN p_provider_id STRING,
    OUT p_total_claims INT,
    OUT p_total_paid DOUBLE,
    OUT p_status_tier STRING
)
LANGUAGE SQL
AS $$
DECLARE
    v_total_paid DOUBLE;
    v_claims_count INT;
BEGIN
    -- 1. Query aggregate statistics
    SELECT COUNT(*), COALESCE(SUM(paid_amount), 0)
    INTO v_claims_count, v_total_paid
    FROM claims
    WHERE provider_id = p_provider_id;

    -- 2. Procedural conditional logic
    IF v_total_paid > 1000000 THEN
        p_status_tier := 'Tier 1 - High Volume';
    ELSEIF v_total_paid > 250000 THEN
        p_status_tier := 'Tier 2 - Medium Volume';
    ELSE
        p_status_tier := 'Tier 3 - Standard';
    END IF;

    -- 3. Assign OUT parameters
    p_total_claims := v_claims_count;
    p_total_paid := v_total_paid;
END;
$$;
```

Let's test this procedural logic directly on our claims warehouse using an interactive Python-procedural wrapper:


In [ ]:
# Interactive Simulation: Procedural Logic with Parameters and Branching

def sp_get_provider_summary(provider_id: str) -> dict:
    """Simulates execution of stored procedure get_provider_summary."""
    con = connect()
    
    # Step 1: Query aggregate values
    result = con.execute('''
        SELECT 
            COUNT(*) AS claim_count,
            COALESCE(SUM(paid_amount), 0) AS total_paid
        FROM claims
        WHERE provider_id = ?
    ''', [provider_id]).fetchone()
    
    claim_count, total_paid = result[0], float(result[1])
    
    # Step 2: Procedural control flow (IF / ELSEIF / ELSE)
    if total_paid > 1_000_000:
        tier = 'Tier 1 - High Volume'
    elif total_paid > 250_000:
        tier = 'Tier 2 - Medium Volume'
    else:
        tier = 'Tier 3 - Standard'
        
    # Step 3: Return OUT parameters
    return {
        "provider_id": provider_id,
        "total_claims": claim_count,
        "total_paid": round(total_paid, 2),
        "status_tier": tier
    }

# Test with a few providers
for pid in ["PRV0151", "PRV0020", "PRV0001"]:
    summary = sp_get_provider_summary(pid)
    print(f"[{summary['provider_id']}] Claims: {summary['total_claims']:<4} | "
          f"Paid: ${summary['total_paid']:>12,.2f} | Tier: {summary['status_tier']}")


---
### Module 2: The Core DE Workhorse - Idempotent Batch Ingestion

The most frequent real-world use case for stored procedures is **Batch ETL Ingestion**.

#### What is Idempotency?
An operation is **idempotent** if running it once produces the exact same result as running it 10 times.
* ❌ **Non-idempotent:** `INSERT INTO target SELECT ... FROM source;` (re-running on pipeline retry duplicates rows!).
* ✅ **Idempotent (Delete + Insert):**
  ```sql
  DELETE FROM target WHERE partition_month = p_month;
  INSERT INTO target SELECT ... FROM source WHERE partition_month = p_month;
  ```
* ✅ **Idempotent (`MERGE INTO`):**
  Matches on primary key; updates if matched, inserts if new.

#### Scenario: `sp_refresh_monthly_kpi(p_month)`
Let's build an ingestion procedure that refreshes the monthly KPI mart for a specific month.


In [ ]:
# Setup: Create target table for our Stored Procedure
con = connect()

con.execute('''
CREATE TEMP TABLE IF NOT EXISTS monthly_kpi_target (
    month DATE PRIMARY KEY,
    total_claims BIGINT,
    approved_claims BIGINT,
    denied_claims BIGINT,
    approval_rate DOUBLE,
    total_paid DOUBLE,
    _last_refreshed_at TIMESTAMP
);
''')

def sp_refresh_monthly_kpi(target_month: str) -> dict:
    """Idempotent partition reload procedure:
    1. Validates input
    2. Deletes existing partition records (prevents duplication)
    3. Inserts fresh aggregate records
    4. Returns execution audit statistics
    """
    con = connect()
    
    # 1. Atomic DELETE existing partition
    deleted_count = con.execute('''
        DELETE FROM monthly_kpi_target 
        WHERE month = ?::DATE
    ''', [target_month]).fetchone()[0]
    
    # 2. Re-compute and INSERT fresh partition
    con.execute('''
        INSERT INTO monthly_kpi_target
        SELECT
            date_trunc('month', claim_date)::DATE AS month,
            COUNT(*) AS total_claims,
            COUNT(*) FILTER (WHERE status = 'Approved') AS approved_claims,
            COUNT(*) FILTER (WHERE status = 'Denied') AS denied_claims,
            ROUND(COUNT(*) FILTER (WHERE status = 'Approved') * 100.0 / NULLIF(COUNT(*), 0), 2) AS approval_rate,
            ROUND(SUM(paid_amount), 2) AS total_paid,
            CURRENT_TIMESTAMP AS _last_refreshed_at
        FROM claims
        WHERE date_trunc('month', claim_date)::DATE = ?::DATE
        GROUP BY 1;
    ''', [target_month])
    
    # 3. Fetch newly inserted row count
    inserted_row = con.execute('''
        SELECT total_claims, approval_rate, total_paid 
        FROM monthly_kpi_target 
        WHERE month = ?::DATE
    ''', [target_month]).fetchone()
    
    return {
        "target_month": target_month,
        "rows_deleted_prior": deleted_count,
        "claims_loaded": inserted_row[0] if inserted_row else 0,
        "approval_rate": inserted_row[1] if inserted_row else 0.0,
        "total_paid": inserted_row[2] if inserted_row else 0.0,
        "status": "SUCCESS"
    }

# Run 1: First load for '2024-05-01'
print("--- RUN 1 (Initial Load) ---")
res1 = sp_refresh_monthly_kpi('2024-05-01')
print(res1)

# Run 2: Re-run immediately on the SAME date (Simulating pipeline retry)
print("\n--- RUN 2 (Pipeline Retry / Idempotency Check) ---")
res2 = sp_refresh_monthly_kpi('2024-05-01')
print(res2)

# Verify table row count: exactly 1 row for that month!
q("SELECT * FROM monthly_kpi_target")


---
### Module 3: Transactions & Error Handling (`BEGIN`, `COMMIT`, `ROLLBACK`)

In production pipelines, a stored procedure often touches **multiple tables**.
For example:
1. Updating account balances or claim payouts.
2. Logging the execution into an `audit_pipeline_runs` table.

#### The Nightmare Scenario: Partial Failure
If step 1 succeeds, but step 2 fails due to a network glitch or syntax error, your database is left in a **corrupt, half-written state**.

This is where **ACID Transactions** are vital:
```sql
BEGIN TRANSACTION;

TRY
    -- Step 1: Update financial records
    UPDATE claims SET status = 'Adjudicated' WHERE claim_id = p_claim_id;
    
    -- Step 2: Log to audit table
    INSERT INTO audit_pipeline_runs (pipeline_name, status) VALUES ('adjudication', 'SUCCESS');
    
    COMMIT; -- All changes permanently saved
CATCH
    ROLLBACK; -- Undo everything! Revert back to the state before BEGIN
    -- Log failure and re-throw
    INSERT INTO audit_pipeline_runs (pipeline_name, status, error) VALUES ('adjudication', 'FAILED', error_message());
END TRY;
```

Let's test this in code with an intentional failure test:


In [ ]:
# Create audit table to track procedure runs
con = connect()
con.execute('''
CREATE TEMP TABLE IF NOT EXISTS pipeline_audit_log (
    run_id VARCHAR PRIMARY KEY,
    pipeline_name VARCHAR,
    records_processed BIGINT,
    status VARCHAR,
    error_message VARCHAR,
    executed_at TIMESTAMP
);
''')

import uuid

def sp_atomic_batch_load(batch_month: str, simulate_failure: bool = False) -> str:
    """Demonstrates atomic transactional execution.
    If simulate_failure=True, rolls back the target table write and logs the error.
    """
    con = connect()
    run_id = f"RUN_{uuid.uuid4().hex[:6]}"
    
    try:
        # Start transaction
        con.execute("BEGIN TRANSACTION;")
        
        # Step 1: Write to business table (Atomic delete + insert)
        con.execute("DELETE FROM monthly_kpi_target WHERE month = ?::DATE;", [batch_month])
        con.execute('''
            INSERT INTO monthly_kpi_target
            SELECT 
                date_trunc('month', claim_date)::DATE,
                COUNT(*),
                COUNT(*) FILTER (WHERE status = 'Approved'),
                COUNT(*) FILTER (WHERE status = 'Denied'),
                ROUND(COUNT(*) FILTER (WHERE status = 'Approved') * 100.0 / COUNT(*), 2),
                ROUND(SUM(paid_amount), 2),
                CURRENT_TIMESTAMP
            FROM claims
            WHERE date_trunc('month', claim_date)::DATE = ?::DATE
            GROUP BY 1;
        ''', [batch_month])
        
        # Simulated failure point
        if simulate_failure:
            raise ValueError("Network timeout connecting to external compliance service!")
            
        # Step 2: Log success in audit table
        con.execute('''
            INSERT INTO pipeline_audit_log VALUES (?, 'monthly_kpi_load', 1, 'SUCCESS', NULL, CURRENT_TIMESTAMP);
        ''', [run_id])
        
        con.execute("COMMIT;")
        return f"SUCCESS: Run {run_id} committed."
        
    except Exception as e:
        # ROLLBACK: All changes made since BEGIN are undone
        con.execute("ROLLBACK;")
        
        # Write failure log in separate transaction
        con.execute('''
            INSERT INTO pipeline_audit_log VALUES (?, 'monthly_kpi_load', 0, 'FAILED', ?, CURRENT_TIMESTAMP);
        ''', [run_id, str(e)])
        
        return f"FAILED & ROLLED BACK: Run {run_id}. Error: {e}"

# Test 1: Successful execution
print(sp_atomic_batch_load('2024-06-01', simulate_failure=False))

# Test 2: Simulated failure (notice rollback!)
print(sp_atomic_batch_load('2024-07-01', simulate_failure=True))

# Check audit log
q("SELECT * FROM pipeline_audit_log ORDER BY executed_at DESC LIMIT 5")


---
### Module 4: Dynamic SQL (`EXECUTE IMMEDIATE`) & Warehouse Maintenance

#### What is Dynamic SQL?
Dynamic SQL is SQL code that is constructed as a text string at runtime and executed dynamically.

In **Databricks SQL** and **Snowflake**, this is done using **`EXECUTE IMMEDIATE`**:
```sql
DECLARE v_sql STRING;
DECLARE v_table_name STRING DEFAULT 'claims_2024_01';

BEGIN
    -- Construct dynamic command (e.g. partition optimization)
    v_sql := 'OPTIMIZE ' || v_table_name || ' ZORDER BY (claim_date, provider_id);';
    EXECUTE IMMEDIATE v_sql;
END;
```

#### When to use Dynamic SQL in Data Engineering:
1. **Maintenance & Vacuuming:** Automating `OPTIMIZE`, `VACUUM`, and index rebuilds across all tables in a catalog.
2. **Metadata-Driven Pipelines:** Looping over an `information_schema.tables` list to archive or validate tables.
3. **Partition Dropping:** Purging partitions older than retention policy (e.g., `retention_months > 36`).

#### ⚠️ Top Interview Warning: SQL Injection!
Never concatenate untrusted user input directly into dynamic queries:
```sql
-- ❌ DANGEROUS (SQL Injection risk):
EXECUTE IMMEDIATE 'DELETE FROM ' || user_input;

-- ✅ SAFE: Validate against allowed whitelist or use parameterized variables:
IF user_input NOT IN ('claims_archive_2023', 'claims_archive_2024') THEN
    RAISE EXCEPTION 'Invalid table name!';
END IF;
```


In [ ]:
# Dynamic SQL Simulation: Metadata-driven table archiver

def sp_archive_monthly_claims(year: int, month: int) -> dict:
    """Dynamically creates a monthly archive table from claims."""
    con = connect()
    
    # 1. Sanitize & construct table name
    target_table = f"claims_archive_{year}_{month:02d}"
    start_date = f"{year}-{month:02d}-01"
    
    # 2. Dynamic DDL/DML string
    sql = f'''
        CREATE OR REPLACE TEMP TABLE {target_table} AS
        SELECT * FROM claims
        WHERE date_trunc('month', claim_date)::DATE = '{start_date}'::DATE;
    '''
    
    # Execute dynamic SQL
    con.execute(sql)
    
    # Inspect newly created archive
    count = con.execute(f"SELECT COUNT(*) FROM {target_table}").fetchone()[0]
    
    return {
        "archive_table": target_table,
        "rows_archived": count,
        "partition_date": start_date
    }

archive_info = sp_archive_monthly_claims(2024, 1)
print("Dynamic SQL executed successfully:", archive_info)
q(f"SELECT claim_id, claim_date, billed_amount, paid_amount FROM {archive_info['archive_table']} LIMIT 5")


---
### Module 5: The #1 Interview Drill - Refactoring Legacy Stored Procs into PySpark

A staple of senior DE interviews is:
> *"We have a 2,000-line Oracle / SQL Server stored procedure with cursors and loops. How do you migrate this to Databricks/PySpark?"*

#### The "Cursor & Loop" Anti-Pattern in Legacy SQL:
In legacy RDBMS systems, developers frequently used **CURSORS** to loop row-by-row:
```sql
-- ❌ LEGACY RDBMS ANTI-PATTERN (Row-by-Row Cursor)
DECLARE claim_cursor CURSOR FOR SELECT claim_id, paid_amount FROM claims;
OPEN claim_cursor;
FETCH NEXT FROM claim_cursor INTO @id, @paid;
WHILE @@FETCH_STATUS = 0
BEGIN
    -- Calculating cumulative total row-by-row
    SET @running_total = @running_total + @paid;
    UPDATE claims SET running_total = @running_total WHERE claim_id = @id;
    FETCH NEXT FROM claim_cursor INTO @id, @paid;
END;
CLOSE claim_cursor;
```

#### Why Cursors Fail in Big Data / Distributed Engines:
* Cursors force single-threaded, sequential row-by-row evaluation on a single CPU core.
* Distributed engines (Spark, Databricks, Snowflake, BigQuery) are built for **set-based, vectorized parallel processing**.

#### The Modern Replacement:
Replace the cursor with a **Window Function** in PySpark or modern SQL!


In [ ]:
# Refactoring Demonstration:
# Moving from row-by-row procedural cursor logic to Vectorized PySpark Windowing!

from pyspark.sql import functions as F, Window

# 1. Load claims
claims_spk = spark.table("claims").filter(F.col("provider_id") == "PRV0151")

# 2. Modern Set-based Vectorized calculation (Zero loops, 100% parallel!)
window_spec = (
    Window.partitionBy("provider_id")
    .orderBy("claim_date", "claim_id")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

refactored_df = (
    claims_spk
    .withColumn("running_total_paid", F.round(F.sum("paid_amount").over(window_spec), 2))
    .select("provider_id", "claim_id", "claim_date", "paid_amount", "running_total_paid")
    .orderBy("claim_date", "claim_id")
)

print("Vectorized PySpark calculation (replaces 50 lines of legacy cursor logic):")
display(refactored_df.limit(10))


---
### 5 Essential Interview Questions on Stored Procedures

1. **Q: What is the difference between a Stored Procedure and a User-Defined Function (UDF)?**
   * **Answer:** A UDF is a pure computation that returns a value/table, takes inputs, and **cannot perform DML** (cannot modify tables or database state). A Stored Procedure is designed for procedural workflows with side effects (`INSERT`, `UPDATE`, `DELETE`, `MERGE`, DDL) and can return multiple result sets or status codes.

2. **Q: How do you achieve idempotency in a stored procedure batch pipeline?**
   * **Answer:** By using either an atomic partition replace (`DELETE FROM target WHERE partition = p_date` followed by `INSERT`), a `MERGE INTO` statement with an update/insert matching condition, or staging table swap. Re-running the procedure must leave the database in the exact same state without duplicate rows.

3. **Q: Why are cursors considered an anti-pattern in modern cloud data warehouses like Databricks or Snowflake?**
   * **Answer:** Cursors process data sequentially row-by-row on a single thread. Modern cloud warehouses and Spark rely on mass parallel, set-based vector operations. Row-by-row cursor logic should be refactored into Window functions, `JOIN`s, or set operations.

4. **Q: When would you use a Python script / Airflow DAG instead of a Stored Procedure?**
   * **Answer:** Use Python/Airflow when the task involves external systems (calling REST APIs, downloading S3/blob files, Slack alerts), requires complex libraries (machine learning, data science), or cross-database orchestration. Use Stored Procedures when the data already resides entirely within the warehouse and operations benefit from pushdown execution.

5. **Q: How are transactions handled inside a stored procedure?**
   * **Answer:** Using `BEGIN TRANSACTION`, `COMMIT`, and `ROLLBACK` enclosed inside a `TRY...CATCH` or exception block. If any step fails, `ROLLBACK` guarantees atomicity so partial, corrupted writes never persist.
